In [0]:
%run "./02-chunking"

In [0]:
%run "./03-embeddings"

In [0]:
%sql
select * from rag_demo.logs.embeddings

####User question
####      ↓
####Convert to embedding
####      ↓
####Send to endpoint
####      ↓
####Endpoint searches index
####      ↓
####Returns top matching chunks

In [0]:
context = " ".join([row[0] for row in results['result']['data_array']])
print(context)

In [0]:
from openai import AzureOpenAI
import os

chat_client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint="https://rahul-mod0jads-eastus2.cognitiveservices.azure.com/",
    api_version="2024-02-01"   # 👈 THIS IS CRITICAL
)

In [0]:
    # Step 1: question
    question = "what were the total pipelines deployed, success and failed?"

    # Step 2: create embedding
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    # Step 3: vector search
    from databricks.vector_search.client import VectorSearchClient

    vs_client = VectorSearchClient()

    index = vs_client.get_index(
        index_name="rag_demo.logs.rag_index"
    )

    results = index.similarity_search(
        query_vector=query_embedding,
        columns=["chunk_text"],
        num_results=5
    )

    # Step 4: build context
    context = " ".join([row[0] for row in results['result']['data_array']])

    # Step 5: LLM call (FINAL)
    response = chat_client.chat.completions.create(
        model="chat-model",
        messages=[
            {
                "role": "system",
                "content": "Answer ONLY based on the given context."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{question}"
            }
        ]
    )

    print("\nFinal Answer:\n", response.choices[0].message.content)

We built a RAG-based system to analyze pipeline logs.

First, logs stored in ADLS were ingested into Databricks.
Since embeddings work better on natural language, we converted structured logs into text format.

Next, we applied time-based chunking (e.g., 30-minute windows) to break large data into smaller segments. This helps in efficient retrieval and avoids token limits when interacting with LLMs.

Each chunk was assigned a unique chunk_id and converted into embeddings using an embedding model. These embeddings represent the semantic meaning of the text.

We stored chunk_text, chunk_id, and embeddings in a Delta table.

On top of this, we created a Vector Search Index using Databricks, which enables fast similarity search over embeddings.

At query time, the user question is converted into an embedding. The vector search compares this query embedding with stored embeddings using similarity metrics and retrieves the top K most relevant chunks.

These retrieved chunks are passed as context to the LLM.

Finally, the LLM generates an answer based only on this retrieved context, which significantly reduces hallucination and ensures responses are grounded in actual data.
